# QE band - spin plotting

QuantumEspressoでエネルギーバンドを計算した場合の、バンド構造とスピンテクスチャの解析スクリプト

#### モジュール

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import re
import os
import ipynbname
NB_NAME = ipynbname.name()

#### データセットの定義

In [ ]:
# データセットの指定（各自の環境に合わせて記入）

dataset = {
    "title": "",
    "band": {
        "file": "",
    },
    "kpath": {
        "ticks": [],
        "labels": [],
    },
    "spin": [
        # {"title": "Sx", "file": ""},
        # {"title": "Sy", "file": ""},
        # {"title": "Sz", "file": ""},
    ],
}


#### QEバンド・スピン期待値ファイル読み込み関数

In [ ]:
# bands.outファイル読み取り（スピン期待値ファイルも同一フォーマットのため共用）
def read_qe_band_file(filename):
    with open(filename, 'r') as f:
        content = f.read()
    header = re.search(r'nbnd=\s*(\d+),\s*nks=\s*(\d+)', content)
    nbnd = int(header.group(1))
    nks  = int(header.group(2))
    numbers = np.fromstring(re.sub(r'&plot[^\n]*/\n', '', content), sep=' ')
    energies = numbers.reshape(nks, 3 + nbnd)[:, 3:]  # k座標3列を除く
    return np.arange(nks), energies  # shape: (nks, nbnd)

#### プロット

In [ ]:
# データセットの指定
dataset = dataset_material_a
title = dataset["title"]

# サブプロットフレームの作成
n_panels = 1 + len(dataset["spin"])
fig, axes = plt.subplots(1, n_panels, figsize=(2.5 * n_panels, 4), sharex=True, sharey=True, layout="constrained")
fig.canvas.header_visible = False

# 全バンドをまとめてLineCollection用のセグメントに変換（高速化のため）
segments = np.stack([np.broadcast_to(kpoints[:, None], bands.shape), bands], axis=-1)
segments = segments.transpose(1, 0, 2)  # (nbnd, nks, 2)
kpoints_tiled = np.broadcast_to(kpoints[:, None], bands.shape).ravel()

# バンド構造の読み込み
kpoints, bands = read_qe_band_file(dataset["band"]["file"])

# スピン表示なしバンドの描画
ax1 = axes[0]
ax1.add_collection(LineCollection(segments, colors='black', linewidths=0.5))
ax1.set_xlim(0, max(kpoints))
ax1.set_ylim(bands.min(), bands.max())
ax1.set_xlabel("K-point")
ax1.set_ylabel("Energy (eV)")
ax1.set_xticks(dataset["kpath"]["ticks"])
ax1.set_xticklabels(dataset["kpath"]["labels"])
ax1.set_title("Band")
ax1.grid(True, linestyle=':')

# 各方向のスピン期待値マップの描画
sm = None
for idx, comp in enumerate(dataset["spin"]):
    ax = axes[idx + 1]
    _, spins = read_qe_band_file(comp["file"])

    ax.add_collection(LineCollection(segments, colors='black', linewidths=0.3, alpha=0.3))
    sm = ax.scatter(kpoints_tiled, bands.ravel(), c=spins.ravel(), cmap='seismic', vmin=-0.5, vmax=0.5, s=3, linewidths=0)

    ax.set_xlabel("K-point")
    ax.set_xticks(dataset["kpath"]["ticks"])
    ax.set_xticklabels(dataset["kpath"]["labels"])
    ax.set_title(comp["title"])
    ax.grid(True, linestyle=':')

fig.suptitle(title, fontsize=16)
fig.colorbar(sm, ax=axes[1:], location="right", shrink=1.0, ticks=[-0.5, -0.25, 0, 0.25, 0.5], label="Spin expectation value")

# 画像として保存
save_dir = f"./{NB_NAME}_save"
os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}/{re.sub(r'[^\w\-]+', '_', title)}.png", dpi=300, bbox_inches="tight")